In [1]:
import pandas as pd
import glob
import os

# =========================
# CONFIG (Convolutional MNIST)
# =========================
BASE_DIR_ROOT = r"C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-CIFAR-100"

PRUNE_LAYERS_OPTIONS = ['CONV', 'FHL', 'SHL', 'FHL+SHL', 'ALL']

PRUNE_LAYER_DIR_MAP = {
    "CONV": "prune_layers_CONV",
    "FHL": "prune_layers_FHL",
    "SHL": "prune_layers_SHL",
    "FHL+SHL": "prune_layers_FHL+SHL",
    "ALL": "prune_layers_ALL"
}

BATCH_DIR_TEMPLATE = "p-percentage_{}\\batch_size_{}"
FILE_PATTERN = "convol_{}_{}_run_*.txt"

BATCH_SIZES = [64,1024]   # extend later if needed
PRUNING_LEVELS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.92, 0.94, 0.96, 0.98, 0.985,0.99,0.995, 1.0] # receont changes 

# =========================
# MAIN AVERAGING LOOP
# =========================
for prune_layer in PRUNE_LAYERS_OPTIONS:

    base_dir = os.path.join(BASE_DIR_ROOT, PRUNE_LAYER_DIR_MAP[prune_layer])

    for bs in BATCH_SIZES:
        for p in PRUNING_LEVELS:

            folder = os.path.join(base_dir, BATCH_DIR_TEMPLATE.format(p, bs))
            files = glob.glob(os.path.join(folder, FILE_PATTERN.format(p, bs)))

            if not files:
                print(f"[WARNING] No files for {prune_layer}, p={p}, bs={bs}")
                continue

            # =========================
            # LOAD ALL RUNS
            # =========================
            dfs = []
            for f in files:
                df = pd.read_csv(f, sep=r"\s+")
                df.columns = df.columns.str.strip()

                df["CE_Train"] = pd.to_numeric(df["CE_Train"], errors="coerce")
                df["CE_TEST"] = pd.to_numeric(df["CE_TEST"], errors="coerce")
                df["Accuracy(%)"] = pd.to_numeric(df["Accuracy(%)"], errors="coerce")

                dfs.append(df)

            all_runs = pd.concat(dfs, ignore_index=True)

            # =========================
            # GROUP + AVERAGE
            # =========================
            avg_df = all_runs.groupby("Batch_Number", as_index=False).agg(
                Avg_CE_Train=("CE_Train", "mean"),
                Avg_CE_Test=("CE_TEST", "mean"),
                Avg_Accuracy=("Accuracy(%)", "mean"),
                Num_Runs=("CE_TEST", "count")
            )

            # =========================
            # SAVE CSV
            # =========================
            out_csv = os.path.join(
                folder,
                f"averaged_runs_conv_{prune_layer}_p_{p}_bs_{bs}.csv"
            )

            avg_df.to_csv(out_csv, index=False)

            print(f"[SAVED] {out_csv}")


[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-CIFAR-100\prune_layers_CONV\p-percentage_0.0\batch_size_64\averaged_runs_conv_CONV_p_0.0_bs_64.csv
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-CIFAR-100\prune_layers_CONV\p-percentage_0.1\batch_size_64\averaged_runs_conv_CONV_p_0.1_bs_64.csv
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-CIFAR-100\prune_layers_CONV\p-percentage_0.2\batch_size_64\averaged_runs_conv_CONV_p_0.2_bs_64.csv
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-CIFAR-100\prune_layers_CONV\p-percentage_0.3\batch_size_64\averaged_runs_conv_CONV_p_0.3_bs_64.csv
[SAVED] C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-CIFAR-100\prune_layers_CONV\p-percentage_0.4\batch_size_64\averaged_runs_conv_CONV_p_0.4_bs_64.csv
[SAVED] C:\Users\Stu